In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import matplotlib.pyplot as plt
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
review = pd.read_csv("https://raw.githubusercontent.com/010shivam/imdb-scraper/refs/heads/main/data/imdb_list.csv")
movies = pd.read_csv("https://raw.githubusercontent.com/010shivam/imdb-scraper/refs/heads/side/data/imdb_reviews.csv")
movies["review"]= movies['review'].str.translate(str.maketrans("","","\r\n\t"))
movies['review']=movies['review'].str.replace("\\","",regex=False)
review.drop(columns={"Unnamed: 0"},inplace=True)

In [ ]:
movies

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
!pip install transformers accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda")
model.to(device)
model.eval()

In [ ]:
import torch.nn.functional as F
import numpy as np

def sentiment_batch(texts, max_length=512):
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits
        probs = F.softmax(logits, dim=-1)
        
    stars = torch.arange(1, 6, device=device).float()
    scores = (probs * stars).sum(dim=1) * 2  # → out of 10

    return scores.cpu().numpy()

In [ ]:
#chunking long reviews
def chunk_text(text, max_tokens=450):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    return [
        tokenizer.decode(tokens[i:i+max_tokens])
        for i in range(0, len(tokens), max_tokens)
    ]

In [ ]:
# full sentiment per review
def get_sentiment(text):
    if not isinstance(text, str) or not text.strip():
        return np.nan

    chunks = chunk_text(text)
    scores = sentiment_batch(chunks)
    return round(scores.mean(), 2)

In [ ]:
from tqdm import tqdm
tqdm.pandas()
movies["bert-rating"] = movies["review"].progress_apply(get_sentiment)

In [ ]:
movies

In [ ]:
movies[movies['id']=="tt28607951"]["review"].iloc[23]

In [ ]:
llm_review = movies.groupby('id',as_index=False)['bert-rating'].mean()

In [ ]:
df =review.merge(llm_review,on='id',how='left').dropna()
df['bert-rating'] = round(df['bert-rating'],2)


In [ ]:
df['deviation']= df['rating'] - df['bert-rating']
df['deviation'] =round(df['deviation'],2)

# Analysis Part

In [ ]:
df

## Distribution of Deviation and range
We see that deviation follows a near to normal distribution. Mean deviation being 0.7 and with std as 0.55.
Deviation ranges from -1 to 2.23 , 

In [ ]:
print(df['deviation'].describe())
sns.kdeplot(df['deviation'],fill=True)

## Deviation correlation with ratings
There is a quite weak correlation between deviation and rating.

In [ ]:
np.corrcoef(df['rating'],df['deviation'])

In [ ]:
sns.scatterplot(df,x='deviation',y='rating')

## How deviation differ in genres ?


In [ ]:
# Need to create a seperate column for mixed genre, single may be insufficient to describe

df["modgenre"]=df['genre'].str.split(",").apply(lambda x : "-".join(x[0:2]))

In [ ]:
df['modgenre'].unique()

In [ ]:
ser = df['modgenre'].value_counts().sort_values(ascending=False)
ser[:10]

In [ ]:
genre_dev =df.groupby('modgenre',as_index=False)['deviation'].mean().sort_values(by='deviation',ascending=False)

In [ ]:
sns.barplot(genre_dev.head(10),y='modgenre',x='deviation')

## How Deviation Varies with Year of Release?

In [ ]:
year_df =df.groupby('year')['deviation'].mean()
year_df


In [ ]:
sns.lineplot(year_df)

## Deviation With Rating

In [ ]:
def bin_rating(x):
    if (x >=5 and x<6):
        return '5-6'
    elif (x>=6 and x<7):
        return '6-7'
    elif (x>=7 and x<8):
        return '7-8'
    elif (x>=8 and x<9):
        return '8-9'
    elif (x>=9 and x<=10):
        return '9-10'
    else:
        return x
df['ratingbins']=df['rating'].apply(bin_rating)
rate_df = df.groupby('ratingbins')['deviation'].mean()
print(rate_df)
sns.barplot(rate_df)

## Are BERT's Rating different from user Ratings

H0 - mean of deviation column is 0

H1- mean of deviation column != 0

**Results:**  The p-value is extremely small close to 0 therefore we reject the null hypotheis , and safely say that rating given my model differ from actual user rating for a movie.

In [ ]:
from statsmodels.stats.weightstats import ztest
from scipy.stats import ttest_1samp

In [ ]:
zstat,pvalue =ztest(df['deviation'],value=0)
print("Zstat: ",zstat)
print("P-value: ",round(pvalue,2))

**Cohens d value**

In [ ]:
df['deviation'].mean()/df['deviation'].std()

In [ ]:
import matplotlib.pyplot as plt

plt.hist(df['deviation'], bins=30)
plt.axvline(0,color='darkblue',linestyle="--")
plt.title("Distribution of Deviation (BERT − Actual)")
plt.show()


## An ANOVA test on variance between genres and ratings

**Genre:** Weak noise, and p value >0.05 so it not much variance in rating among the genre
**Rating Bins** P value < 0.05 , therefore indicates that the there are some ratings have variance.

Tuckey's Test Reveal : Model variance in rating is mostly between 6-7 and 7-8 ratings.

In [ ]:
df["firstgenre"]=df['genre'].str.split(",").str[0]
df['firstgenre'].value_counts()

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

model = ols('deviation ~ C(ratingbins)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print(anova_table)

model = ols('deviation ~ C(firstgenre)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(endog=df['deviation'],
                          groups=df['ratingbins'],
                          alpha=0.05)
print(tukey)


In [ ]:

sns.boxplot(x='ratingbins', y='deviation', data=df)
plt.axhline(0, color='gray', linestyle='--')
plt.title("BERT Deviation by Rating Bin")
plt.show()
